In [1]:
   !pip install requests beautifulsoup4 pandas numpy matplotlib

In [4]:
"""
Assignment 2 - The Bookstore Intelligence Mission
Scrapes https://books.toscrape.com/, builds a DataFrame, saves competitor_books.csv,
answers the 7 analysis questions, and solves the CEO GBP 100 challenge.

Usage:
    pip install requests beautifulsoup4 pandas numpy matplotlib
    python bookstore_scraper.py            # default: 5 pages (Bonus 1)
    python bookstore_scraper.py 3          # minimum requirement: 3 pages
    python bookstore_scraper.py all        # BOSS LEVEL: every catalogue page
"""
import sys
import time
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
PAGE_URL = BASE_URL + "catalogue/page-{}.html"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
BUDGET_GBP = 100


# ---------------------------------------------------------------- scraping
def parse_book(card, page_url):
    """Extract one book's data from an <article class='product_pod'> tag."""
    link = card.h3.a
    title = link["title"]                                   # full title (text is truncated)
    price_text = card.find("p", class_="price_color").get_text(strip=True)
    price = float(price_text.replace("£", "").replace("Â", ""))
    rating_word = card.find("p", class_="star-rating")["class"][1]   # e.g. ['star-rating','Three']
    rating = RATING_MAP[rating_word]
    availability = card.find("p", class_="instock availability").get_text(strip=True)
    url = urljoin(page_url, link["href"])                   # Bonus 2
    return {"Title": title, "Price": price, "Rating": rating,
            "Availability": availability, "URL": url}


def scrape_page(session, page_number):
    """Scrape a single catalogue page. Returns (records, has_page)."""
    url = BASE_URL if page_number == 1 else PAGE_URL.format(page_number)
    if page_number == 1:
        url = PAGE_URL.format(1)
    resp = session.get(url, timeout=15)
    if resp.status_code == 404:
        return [], False
    resp.raise_for_status()
    resp.encoding = "utf-8"                                 # avoids the 'Â£' encoding glitch
    soup = BeautifulSoup(resp.text, "html.parser")
    cards = soup.find_all("article", class_="product_pod")
    return [parse_book(c, url) for c in cards], True


def scrape_books(max_pages=None):
    """Loop through catalogue pages (max_pages=None -> all pages)."""
    records, page = [], 1
    with requests.Session() as session:
        while max_pages is None or page <= max_pages:
            page_records, ok = scrape_page(session, page)
            if not ok:
                break
            records.extend(page_records)
            print(f"Page {page}: {len(page_records)} books")
            page += 1
            time.sleep(0.5)                                 # be polite to the server
    return pd.DataFrame(records)


# ------------------------------------------------------------- CEO challenge
def best_purchase(df, budget=BUDGET_GBP):
    """0/1 knapsack: maximise total stars with total cost <= budget.
    Prices are converted to integer pence so the DP is exact.
    Ties on stars are broken by lowest total cost."""
    costs = (df["Price"] * 100).round().astype(int).to_numpy()
    values = df["Rating"].to_numpy()
    cap = int(budget * 100)
    n = len(df)

    NEG = -10**9
    dp = np.full((n + 1, cap + 1), NEG, dtype=np.int64)
    dp[0, 0] = 0                                            # dp[i][c] = best stars, spending EXACTLY c
    for i in range(1, n + 1):
        dp[i] = dp[i - 1]
        c, v = costs[i - 1], values[i - 1]
        if c <= cap:
            dp[i, c:] = np.maximum(dp[i - 1, c:], dp[i - 1, :cap + 1 - c] + v)

    best_stars = dp[n].max()
    best_cost = int(np.where(dp[n] == best_stars)[0].min())  # cheapest way to reach best_stars

    chosen, c = [], best_cost                                # backtrack
    for i in range(n, 0, -1):
        if dp[i, c] != dp[i - 1, c]:
            chosen.append(i - 1)
            c -= costs[i - 1]
    return df.iloc[sorted(chosen)], int(best_stars), best_cost / 100


# ---------------------------------------------------------------------- main
def main():
    arg = "all" # 3 = minimum, 5 = bonus, "all" = boss level
    max_pages = None if arg.lower() == "all" else int(arg)

    df = scrape_books(max_pages)
    df.to_csv("competitor_books.csv", index=False, encoding="utf-8")
    print(f"\nScraped {len(df)} books -> competitor_books.csv")
    print(df.head(), "\n")
    print(df.dtypes, "\n")

    # ---- Mission 4: the seven intelligence questions
    priciest = df.loc[df["Price"].idxmax()]
    cheapest = df.loc[df["Price"].idxmin()]
    in_stock_pct = df["Availability"].str.contains("In stock", case=False).mean() * 100
    five_star = df[df["Rating"] == 5]

    print("=== INTELLIGENCE ANALYSIS ===")
    print(f"1. Most expensive : {priciest['Title']} (£{priciest['Price']:.2f})")
    print(f"2. Cheapest       : {cheapest['Title']} (£{cheapest['Price']:.2f})")
    print(f"3. Average price  : £{df['Price'].mean():.2f}")
    print(f"4. 5-star books   : {len(five_star)}")
    print(f"5. In stock       : {in_stock_pct:.1f}%")
    print("6. Top 5 highest-rated (ties broken by lower price):")
    print(df.sort_values(["Rating", "Price"], ascending=[False, True])
            .head(5)[["Title", "Rating", "Price"]].to_string(index=False))
    if len(five_star):
        c5 = five_star.loc[five_star["Price"].idxmin()]
        print(f"7. Cheapest 5-star: {c5['Title']} (£{c5['Price']:.2f})")

    # ---- Bonus 3 + 4: rating distribution and bar chart
    counts = df["Rating"].value_counts().sort_index()
    print("\nBooks per rating level:\n", counts.to_string())
    try:
        import matplotlib
        matplotlib.use("Agg")
        ax = counts.plot(kind="bar", color="steelblue", edgecolor="black")
        ax.set_xlabel("Star rating")
        ax.set_ylabel("Number of books")
        ax.set_title("Competitor books by star rating")
        ax.figure.tight_layout()
        ax.figure.savefig("rating_counts.png", dpi=150)
        print("Chart saved -> rating_counts.png")
    except ImportError:
        print("matplotlib not installed - skipping chart")

    # ---- Final challenge: the CEO has GBP 100
    picks, stars, cost = best_purchase(df)
    print(f"\n=== CEO £{BUDGET_GBP} CHALLENGE ===")
    print(picks[["Title", "Price", "Rating"]].to_string(index=False))
    print(f"Books: {len(picks)} | Total cost: £{cost:.2f} | Total rating score: {stars}")


if __name__ == "__main__":
    main()

Page 1: 20 books
Page 2: 20 books
Page 3: 20 books
Page 4: 20 books
Page 5: 20 books
Page 6: 20 books
Page 7: 20 books
Page 8: 20 books
Page 9: 20 books
Page 10: 20 books
Page 11: 20 books
Page 12: 20 books
Page 13: 20 books
Page 14: 20 books
Page 15: 20 books
Page 16: 20 books
Page 17: 20 books
Page 18: 20 books
Page 19: 20 books
Page 20: 20 books
Page 21: 20 books
Page 22: 20 books
Page 23: 20 books
Page 24: 20 books
Page 25: 20 books
Page 26: 20 books
Page 27: 20 books
Page 28: 20 books
Page 29: 20 books
Page 30: 20 books
Page 31: 20 books
Page 32: 20 books
Page 33: 20 books
Page 34: 20 books
Page 35: 20 books
Page 36: 20 books
Page 37: 20 books
Page 38: 20 books
Page 39: 20 books
Page 40: 20 books
Page 41: 20 books
Page 42: 20 books
Page 43: 20 books
Page 44: 20 books
Page 45: 20 books
Page 46: 20 books
Page 47: 20 books
Page 48: 20 books
Page 49: 20 books
Page 50: 20 books

Scraped 1000 books -> competitor_books.csv
                                   Title  Price  Rating Availabil

In [5]:
   from google.colab import files
   files.download("competitor_books.csv")
   files.download("rating_counts.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>